In [1]:
import os
import time
import requests
import pandas as pd
from datetime import datetime

# ============================================================================
# RUTAS - MODIFICA SEGÚN TU ESTRUCTURA
# ============================================================================
BASE_DIR = r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF"

# CSV principal (entrada/salida)
RUTA_DF_MERGED = os.path.join(BASE_DIR, "data", "csv", "df_merged.csv")

# CSV de market cap total histórico (el que descargaste de CoinGecko web)
RUTA_TOTAL_MCAP = os.path.join(BASE_DIR, "data", "csv", "raw", "CoinGecko-GlobalCryptoMktCap-2026-05-02.csv")

# Opcionales (para futuro)
# RUTA_INFLATION = os.path.join(BASE_DIR, "data", "csv", "raw", "inflation.csv")
# RUTA_FED_RATE = os.path.join(BASE_DIR, "data", "csv", "raw", "fed_rate.csv")

# ============================================================================
# APIS
# ============================================================================
COINGECKO_BASE = "https://api.coingecko.com/api/v3"
ALTERNATIVE_BASE = "https://api.alternative.me/fng/"

# Rate limit (free tier: ~10-30 calls/min)
PAUSA_API = 2.5

print("✓ Configuración cargada")
print(f"  df_merged     : {RUTA_DF_MERGED}")
print(f"  Total mcap CSV: {RUTA_TOTAL_MCAP}")

✓ Configuración cargada
  df_merged     : C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged.csv
  Total mcap CSV: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\raw\CoinGecko-GlobalCryptoMktCap-2026-05-02.csv


In [2]:
print("📂 Cargando df_merged.csv...")

df_merged = pd.read_csv(RUTA_DF_MERGED, parse_dates=["date"], index_col="date")
df_merged.sort_index(inplace=True)

ultima_fecha = df_merged.index.max().normalize()
hoy = pd.Timestamp.now().normalize()

print(f"  ✓ Cargado: {len(df_merged)} registros")
print(f"  📅 Última fecha: {ultima_fecha.date()}")
print(f"  📅 Hoy        : {hoy.date()}")
print(f"\n  Columnas ({len(df_merged.columns)}):")
print(f"  {df_merged.columns.tolist()}")

df_merged.tail(3)

📂 Cargando df_merged.csv...
  ✓ Cargado: 3042 registros
  📅 Última fecha: 2026-05-31
  📅 Hoy        : 2026-06-02

  Columnas (19):
  ['btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'eth_open', 'eth_high', 'eth_low', 'eth_close', 'eth_volume', 'btc_dominance', 'eth_dominance', 'alt_dominance', 'fear_greed', 'FearGreed_Label', 'inflation', 'btc_mcap', 'eth_mcap', 'fed_rate']


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2026-05-29,73282.400246,73282.400246,73282.400246,73282.400246,3.219164e+10,2000.340729,2000.340729,2000.340729,2000.340729,1.283154e+10,0.577033,0.094954,0.328013,23.0,Extreme Fear,2.7,1.470013e+12,2.418977e+11,3.84
2026-05-30,73547.208448,73547.208448,73547.208448,73547.208448,3.066212e+10,2017.673674,2017.673674,2017.673674,2017.673674,1.227882e+10,0.574615,0.094943,0.330442,23.0,Extreme Fear,2.7,1.473486e+12,2.434628e+11,3.84
2026-05-31,73732.916640,73732.916640,73732.916640,73732.916640,1.679018e+10,2015.840214,2015.840214,2015.840214,2015.840214,6.557357e+09,0.572863,0.094318,0.332819,28.0,Fear,2.7,1.476834e+12,2.431507e+11,3.84


In [3]:
dias_faltantes = (hoy - ultima_fecha).days

print(f"📊 Días a descargar: {dias_faltantes}")

if dias_faltantes <= 0:
    print("⚠️  El CSV ya está actualizado. No hay nada que hacer.")
    raise SystemExit("CSV actualizado")

📊 Días a descargar: 2


In [4]:
def ajustar_dias_coingecko(dias_necesarios):
    """
    CoinGecko free tier solo acepta valores discretos en /market_chart.
    Redondea al siguiente valor válido.
    """
    valores_validos = [1, 7, 14, 30, 90, 180, 365]
    for v in valores_validos:
        if dias_necesarios <= v:
            return v
    return 365

def descargar_market_chart(coin_id, dias):
    """
    Descarga price, volume, market_cap de CoinGecko /market_chart.
    Devuelve DataFrame con ['close', 'volume', 'market_cap'] indexado por fecha.
    """
    url = f"{COINGECKO_BASE}/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": dias, "interval": "daily"}
    
    print(f"  → Descargando {coin_id}...")
    r = requests.get(url, params=params, timeout=30)
    
    if r.status_code != 200:
        raise RuntimeError(f"Error {r.status_code}: {r.text[:200]}")
    
    data = r.json()
    
    # Parsear datos
    df_price = pd.DataFrame(data["prices"], columns=["ts", "close"])
    df_vol = pd.DataFrame(data["total_volumes"], columns=["ts", "volume"])
    df_mcap = pd.DataFrame(data["market_caps"], columns=["ts", "market_cap"])
    
    # Convertir timestamps a fechas
    for df in (df_price, df_vol, df_mcap):
        df["date"] = pd.to_datetime(df["ts"], unit="ms").dt.normalize()
        df.drop(columns="ts", inplace=True)
        df.set_index("date", inplace=True)
    
    # Join
    df = df_price.join(df_vol).join(df_mcap)
    
    # Deduplicar (por si hay varios timestamps por día)
    df = df.groupby(df.index).last()
    
    return df

def descargar_fear_greed(limit_dias):
    """
    Descarga Fear & Greed histórico de Alternative.me.
    """
    print(f"  → Descargando Fear & Greed...")
    url = ALTERNATIVE_BASE
    params = {"limit": limit_dias, "format": "json"}
    
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Error {r.status_code}")
    
    data = r.json()["data"]
    df = pd.DataFrame(data)
    
    df["date"] = pd.to_datetime(df["timestamp"].astype(int), unit="s").dt.normalize()
    df["fear_greed"] = df["value"].astype(int)
    df["FearGreed_Label"] = df["value_classification"]
    
    return df[["date", "fear_greed", "FearGreed_Label"]].set_index("date").sort_index()

print("✓ Funciones definidas")

✓ Funciones definidas


In [5]:
# Ajustar días al valor válido más cercano
dias_validos = ajustar_dias_coingecko(dias_faltantes + 2)
print(f"🌐 Solicitando {dias_validos} días a CoinGecko (cubre {dias_faltantes} + margen)\n")

# Bitcoin
df_btc = descargar_market_chart("bitcoin", dias_validos)
df_btc.columns = ["btc_close", "btc_volume", "btc_market_cap"]
print(f"  ✓ Bitcoin: {len(df_btc)} registros ({df_btc.index.min().date()} → {df_btc.index.max().date()})")
time.sleep(PAUSA_API)

# Ethereum
df_eth = descargar_market_chart("ethereum", dias_validos)
df_eth.columns = ["eth_close", "eth_volume", "eth_market_cap"]
print(f"  ✓ Ethereum: {len(df_eth)} registros ({df_eth.index.min().date()} → {df_eth.index.max().date()})")
time.sleep(PAUSA_API)

print(f"\n✓ Descarga completada")

🌐 Solicitando 7 días a CoinGecko (cubre 2 + margen)

  → Descargando bitcoin...
  ✓ Bitcoin: 7 registros (2026-05-27 → 2026-06-02)
  → Descargando ethereum...
  ✓ Ethereum: 7 registros (2026-05-27 → 2026-06-02)

✓ Descarga completada


In [6]:
df_feargreed = descargar_fear_greed(limit_dias=dias_validos)
print(f"  ✓ Fear & Greed: {len(df_feargreed)} registros ({df_feargreed.index.min().date()} → {df_feargreed.index.max().date()})")

print(f"\n✓ Fear & Greed descargado")

  → Descargando Fear & Greed...
  ✓ Fear & Greed: 7 registros (2026-05-27 → 2026-06-02)

✓ Fear & Greed descargado


In [7]:
print("📂 Cargando CSV de market cap total histórico...")

# Cargar CSV
df_total_mcap = pd.read_csv(RUTA_TOTAL_MCAP)

# Convertir timestamp de milisegundos a fecha
df_total_mcap["date"] = pd.to_datetime(df_total_mcap["snapped_at"], unit="ms").dt.normalize()
df_total_mcap = df_total_mcap[["date", "market_cap", "total_volume"]]
df_total_mcap.set_index("date", inplace=True)
df_total_mcap.sort_index(inplace=True)

# Renombrar para claridad
df_total_mcap.rename(columns={"market_cap": "total_market_cap"}, inplace=True)

print(f"  ✓ Cargado: {len(df_total_mcap)} registros")
print(f"  📅 Rango: {df_total_mcap.index.min().date()} → {df_total_mcap.index.max().date()}")
print(f"  📊 Último valor: ${df_total_mcap['total_market_cap'].iloc[-1]:,.0f}")

df_total_mcap.tail(3)

📂 Cargando CSV de market cap total histórico...
  ✓ Cargado: 4763 registros
  📅 Rango: 2013-04-29 → 2026-06-02
  📊 Último valor: $2,523,631,545,193


,total_market_cap,total_volume
date,,
2026-05-31,2.577954e+12,5.821959e+10
2026-06-01,2.574987e+12,5.630698e+10
2026-06-02,2.523632e+12,1.179892e+11


In [8]:
print("🔧 Creando OHLC desde close (limitación API gratuita)...")

# Para BTC y ETH: open = high = low = close
for activo in ["btc", "eth"]:
    close_col = f"{activo}_close"
    
    df_btc[f"{activo}_open"] = df_btc[close_col] if activo == "btc" else None
    df_btc[f"{activo}_high"] = df_btc[close_col] if activo == "btc" else None
    df_btc[f"{activo}_low"] = df_btc[close_col] if activo == "btc" else None
    
    if activo == "eth":
        df_eth["eth_open"] = df_eth["eth_close"]
        df_eth["eth_high"] = df_eth["eth_close"]
        df_eth["eth_low"] = df_eth["eth_close"]

# Reordenar columnas: OHLCV
df_btc = df_btc[["btc_open", "btc_high", "btc_low", "btc_close", "btc_volume", "btc_market_cap"]]
df_eth = df_eth[["eth_open", "eth_high", "eth_low", "eth_close", "eth_volume", "eth_market_cap"]]

print("  ✓ OHLC creado")

🔧 Creando OHLC desde close (limitación API gratuita)...
  ✓ OHLC creado


In [9]:
print("🔧 Combinando datos y calculando dominancias...\n")

# 1. Combinar BTC + ETH
df_nuevo = df_btc.join(df_eth, how="outer")
print(f"  ✓ BTC + ETH combinados: {len(df_nuevo)} registros")

# 2. Join con market cap total
df_nuevo = df_nuevo.join(df_total_mcap[["total_market_cap"]], how="left")

# 3. CALCULAR DOMINANCIAS DÍA A DÍA
print(f"\n  🧮 Calculando dominancias día a día...")

dominancias = []

for fecha, row in df_nuevo.iterrows():
    btc_mcap = row["btc_market_cap"]
    eth_mcap = row["eth_market_cap"]
    total_mcap = row["total_market_cap"]
    
    if pd.notna(total_mcap) and total_mcap > 0:
        # Calcular market cap de altcoins
        alt_mcap = total_mcap - btc_mcap - eth_mcap
        
        # Calcular dominancias
        btc_dom = btc_mcap / total_mcap
        eth_dom = eth_mcap / total_mcap
        alt_dom = alt_mcap / total_mcap
    else:
        # Si no hay total_mcap, usar NaN (luego forward-fill)
        btc_dom = eth_dom = alt_dom = None
    
    dominancias.append({
        "btc_dominance": btc_dom,
        "eth_dominance": eth_dom,
        "alt_dominance": alt_dom
    })

# Añadir dominancias al DataFrame
df_dominancias = pd.DataFrame(dominancias, index=df_nuevo.index)
df_nuevo = df_nuevo.join(df_dominancias)

# Forward-fill para fechas sin total_market_cap
df_nuevo["btc_dominance"] = df_nuevo["btc_dominance"].ffill()
df_nuevo["eth_dominance"] = df_nuevo["eth_dominance"].ffill()
df_nuevo["alt_dominance"] = df_nuevo["alt_dominance"].ffill()

print(f"  ✓ Dominancias calculadas")
print(f"\n  📊 Estadísticas de dominancias:")
print(f"     BTC: {df_nuevo['btc_dominance'].iloc[-1]:.4f} ({df_nuevo['btc_dominance'].iloc[-1]*100:.2f}%)")
print(f"     ETH: {df_nuevo['eth_dominance'].iloc[-1]:.4f} ({df_nuevo['eth_dominance'].iloc[-1]*100:.2f}%)")
print(f"     ALT: {df_nuevo['alt_dominance'].iloc[-1]:.4f} ({df_nuevo['alt_dominance'].iloc[-1]*100:.2f}%)")

# 4. Añadir Fear & Greed
df_nuevo = df_nuevo.join(df_feargreed, how="left")
df_nuevo["fear_greed"] = df_nuevo["fear_greed"].ffill()
df_nuevo["FearGreed_Label"] = df_nuevo["FearGreed_Label"].ffill()
print(f"\n  ✓ Fear & Greed añadido")

# 5. Añadir inflation y fed_rate (forward-fill del último valor)
df_nuevo["inflation"] = df_merged["inflation"].iloc[-1]
df_nuevo["fed_rate"] = df_merged["fed_rate"].iloc[-1]
print(f"  ✓ Inflation y Fed Rate (forward-filled)")

# 6. Renombrar market_cap a mcap
df_nuevo.rename(columns={
    "btc_market_cap": "btc_mcap",
    "eth_market_cap": "eth_mcap"
}, inplace=True)

# 7. Drop total_market_cap (no está en df_merged)
df_nuevo.drop(columns=["total_market_cap", "total_volume"], inplace=True, errors="ignore")

# 8. Reordenar columnas para match con df_merged
columnas_orden = [
    "btc_open", "btc_high", "btc_low", "btc_close", "btc_volume",
    "eth_open", "eth_high", "eth_low", "eth_close", "eth_volume",
    "btc_dominance", "eth_dominance", "alt_dominance",
    "fear_greed", "FearGreed_Label",
    "inflation",
    "btc_mcap", "eth_mcap",
    "fed_rate"
]

df_nuevo = df_nuevo[columnas_orden]

print(f"\n✓ DataFrame nuevo formateado")
print(f"  Columnas: {len(df_nuevo.columns)} (match con df_merged)")
print(f"  NaN totales: {df_nuevo.isna().sum().sum()}")

df_nuevo.head()

🔧 Combinando datos y calculando dominancias...

  ✓ BTC + ETH combinados: 7 registros

  🧮 Calculando dominancias día a día...
  ✓ Dominancias calculadas

  📊 Estadísticas de dominancias:
     BTC: 0.5374 (53.74%)
     ETH: 0.0920 (9.20%)
     ALT: 0.3706 (37.06%)

  ✓ Fear & Greed añadido
  ✓ Inflation y Fed Rate (forward-filled)

✓ DataFrame nuevo formateado
  Columnas: 19 (match con df_merged)
  NaN totales: 0


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2026-05-27,75824.063325,75824.063325,75824.063325,75824.063325,3.915969e+10,2070.833946,2070.833946,2070.833946,2070.833946,1.428936e+10,0.580184,0.095444,0.324371,25,Extreme Fear,2.7,1.519006e+12,2.498874e+11,3.84
2026-05-28,74352.699280,74352.699280,74352.699280,74352.699280,3.645656e+10,2022.506249,2022.506249,2022.506249,2022.506249,1.326619e+10,0.577875,0.094694,0.327431,22,Extreme Fear,2.7,1.489193e+12,2.440285e+11,3.84
2026-05-29,73539.841896,73539.841896,73539.841896,73539.841896,3.948262e+10,2007.562411,2007.562411,2007.562411,2007.562411,1.682961e+10,0.576800,0.094849,0.328351,23,Extreme Fear,2.7,1.473191e+12,2.422517e+11,3.84
2026-05-30,73382.718314,73382.718314,73382.718314,73382.718314,3.550017e+10,2012.230103,2012.230103,2012.230103,2012.230103,1.338978e+10,0.575691,0.095066,0.329243,23,Extreme Fear,2.7,1.470438e+12,2.428178e+11,3.84
2026-05-31,73751.067873,73751.067873,73751.067873,73751.067873,1.804189e+10,2019.175627,2019.175627,2019.175627,2019.175627,6.649284e+09,0.573201,0.094524,0.332275,28,Fear,2.7,1.477685e+12,2.436784e+11,3.84


In [10]:
print("🔧 Filtrando solo días nuevos...")

df_nuevo_filtrado = df_nuevo[df_nuevo.index > ultima_fecha].copy()

print(f"  ✓ Días nuevos: {len(df_nuevo_filtrado)}")
print(f"  📅 Rango: {df_nuevo_filtrado.index.min().date()} → {df_nuevo_filtrado.index.max().date()}")

if df_nuevo_filtrado.empty:
    print("⚠️  No hay días nuevos tras filtrar")
    raise SystemExit("Sin días nuevos")

df_nuevo_filtrado.tail()

🔧 Filtrando solo días nuevos...
  ✓ Días nuevos: 2
  📅 Rango: 2026-06-01 → 2026-06-02


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2026-06-01,73593.371197,73593.371197,73593.371197,73593.371197,1.675375e+10,2003.971930,2003.971930,2003.971930,2003.971930,7.997871e+09,0.572650,0.093922,0.333428,29,Fear,2.7,1.474567e+12,2.418485e+11,3.84
2026-06-02,67674.396741,67674.396741,67674.396741,67674.396741,5.790313e+10,1926.351414,1926.351414,1926.351414,1926.351414,1.917185e+10,0.537374,0.091992,0.370634,23,Extreme Fear,2.7,1.356135e+12,2.321529e+11,3.84


In [11]:
print("🔗 Mergeando con df_merged...")

# Concatenar
df_actualizado = pd.concat([df_merged, df_nuevo_filtrado])

# Eliminar duplicados (keep last por si actualizaste algo)
df_actualizado = df_actualizado[~df_actualizado.index.duplicated(keep="last")]

# Ordenar por fecha
df_actualizado.sort_index(inplace=True)

print(f"  ✓ Merge completado")
print(f"  📊 Total registros: {len(df_actualizado)} ({len(df_merged)} antiguos + {len(df_nuevo_filtrado)} nuevos)")
print(f"  📅 Última fecha: {df_actualizado.index.max().date()}")

df_actualizado.tail()

🔗 Mergeando con df_merged...
  ✓ Merge completado
  📊 Total registros: 3044 (3042 antiguos + 2 nuevos)
  📅 Última fecha: 2026-06-02


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2026-05-29,73282.400246,73282.400246,73282.400246,73282.400246,3.219164e+10,2000.340729,2000.340729,2000.340729,2000.340729,1.283154e+10,0.577033,0.094954,0.328013,23.0,Extreme Fear,2.7,1.470013e+12,2.418977e+11,3.84
2026-05-30,73547.208448,73547.208448,73547.208448,73547.208448,3.066212e+10,2017.673674,2017.673674,2017.673674,2017.673674,1.227882e+10,0.574615,0.094943,0.330442,23.0,Extreme Fear,2.7,1.473486e+12,2.434628e+11,3.84
2026-05-31,73732.916640,73732.916640,73732.916640,73732.916640,1.679018e+10,2015.840214,2015.840214,2015.840214,2015.840214,6.557357e+09,0.572863,0.094318,0.332819,28.0,Fear,2.7,1.476834e+12,2.431507e+11,3.84
2026-06-01,73593.371197,73593.371197,73593.371197,73593.371197,1.675375e+10,2003.971930,2003.971930,2003.971930,2003.971930,7.997871e+09,0.572650,0.093922,0.333428,29.0,Fear,2.7,1.474567e+12,2.418485e+11,3.84
2026-06-02,67674.396741,67674.396741,67674.396741,67674.396741,5.790313e+10,1926.351414,1926.351414,1926.351414,1926.351414,1.917185e+10,0.537374,0.091992,0.370634,23.0,Extreme Fear,2.7,1.356135e+12,2.321529e+11,3.84


In [12]:
print("💾 Guardando CSV actualizado...")

# Backup del CSV antiguo
import shutil
backup_path = RUTA_DF_MERGED.replace(".csv", f"_backup_{hoy.strftime('%Y%m%d')}.csv")
shutil.copy(RUTA_DF_MERGED, backup_path)
print(f"  ✓ Backup guardado: {backup_path}")

# Guardar CSV actualizado
df_actualizado.to_csv(RUTA_DF_MERGED)
print(f"  ✓ CSV actualizado guardado: {RUTA_DF_MERGED}")
print(f"\n📊 Resumen final:")
print(f"   Total registros: {len(df_actualizado)}")
print(f"   Rango: {df_actualizado.index.min().date()} → {df_actualizado.index.max().date()}")
print(f"   Días añadidos: {len(df_nuevo_filtrado)}")

💾 Guardando CSV actualizado...
  ✓ Backup guardado: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged_backup_20260602.csv
  ✓ CSV actualizado guardado: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged.csv

📊 Resumen final:
   Total registros: 3044
   Rango: 2018-02-01 → 2026-06-02
   Días añadidos: 2
